<a href="https://colab.research.google.com/github/bymahtab/clipping/blob/main/liveclip_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📱 liveclip on your phone (free)

Turn any GTA stream into ready-to-post vertical clips: **streamer's face on top, gameplay below, captions on.** It runs on Google's free computers, so you only need a phone and a Google account. No laptop, no install.

**How to use it**
1. Paste a YouTube or Twitch link in the box below. It can be a **live stream** or a **finished video**.
2. Tap the **▶ play button**. If Colab says *"This notebook was not authored by Google"*, tap **Run anyway**.
3. Keep this tab open. Your clips show up below with a **Download** button.

**On an iPhone?** iPhone browsers can't save videos from this page. Tick **save_to_google_drive** before you tap play: every clip also goes to your Google Drive (My Drive → liveclip). Then in the Google Drive app: ⋯ → **Send a copy** → **Save Video**.

Live stream: it clips the moments when the chat explodes. Finished video: it finds the loudest moments (shouting, hype, big reactions). It finds the streamer's facecam by itself; if it picks the wrong spot, choose the corner in **facecam**.

> **Only clip streamers who allow clipping** (or who run an official clipping program). Clips without permission get copyright strikes and no payout.

Made by [@aaditya.knows](https://instagram.com/aaditya.knows) · code: [github.com/Aadi1659/liveclip](https://github.com/Aadi1659/liveclip)

In [ ]:
#@title ▶ Paste the link, then tap play
link = ""  #@param {type:"string"}
facecam = "find it for me"  #@param ["find it for me", "top left", "top right", "bottom left", "bottom right", "no facecam"]
clips = 5  #@param {type:"slider", min:1, max:15, step:1}
save_to_google_drive = False  #@param {type:"boolean"}
#@markdown 📱 **Have an iPhone? Tick save_to_google_drive.** iPhone browsers can't save the clips from this page, so they go to your Google Drive instead.

import os, re, signal, subprocess, sys, base64, shutil
from IPython.display import display, HTML

url = link.strip()
if not re.search(r"(youtube\.com|youtu\.be|twitch\.tv)/", url):
    raise SystemExit("Paste a YouTube or Twitch link in the link box first.")

# 1) setup (first run only, about a minute)
if not os.path.exists("/content/liveclip/liveclip.py"):
    print("Setting up (first run only, about a minute)…")
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/Aadi1659/liveclip.git", "/content/liveclip"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "streamlink>=7.0", "faster-whisper>=1.0", "pillow>=10.0",
                    "yt-dlp", "opencv-python-headless<5"], check=True)
    print("Setup done.")
else:
    subprocess.run(["git", "-C", "/content/liveclip", "pull", "-q"])  # pick up fixes on a reused runtime

# iPhone browsers can't save the Download button's video, so clips also go to Google Drive (ask for access now, not after the wait)
drive_dir = None
if save_to_google_drive:
    from google.colab import drive
    print("Connecting your Google Drive: tap Connect and pick your Google account.")
    drive.mount("/content/drive")
    drive_dir = "/content/drive/MyDrive/liveclip"; os.makedirs(drive_dir, exist_ok=True)

face = {"find it for me": "auto", "no facecam": "none"}.get(facecam, facecam.replace(" ", "-"))
out_dir = "/content/clips"
# good defaults for everyone: 30 s clips, a live stream is watched for up to an hour, Hinglish captions
cmd = [sys.executable, "-u", "/content/liveclip/liveclip.py", url, "--mode", "auto", "--face", face, "--out", out_dir,
       "--pre", "24", "--post", "6", "--spike", "1.7", "--duration", "3600", "--max-clips", str(clips),
       "--captions", "hinglish", "--model", "auto", "--device", "auto"]


def show(path):
    """Show the clip in the notebook with a player you can download from."""
    b64 = base64.b64encode(open(path, "rb").read()).decode()
    name = os.path.basename(path)
    display(HTML(f'<p style="font:600 15px sans-serif;margin:14px 0 6px">🎬 {name}</p>'
                 f'<video controls playsinline style="width:100%;max-width:360px;border-radius:12px" '
                 f'src="data:video/mp4;base64,{b64}"></video>'
                 f'<br><a download="{name}" href="data:video/mp4;base64,{b64}" '
                 f'style="display:inline-block;margin-top:6px;padding:10px 16px;border-radius:10px;'
                 f'background:#e8684a;color:#fff;font:600 15px sans-serif;text-decoration:none">⬇ Download {name}</a>'))
    if drive_dir:
        shutil.copy(path, drive_dir)
        display(HTML('<p style="font:14px sans-serif;margin:6px 0 0">📁 Saved to Google Drive → My Drive → liveclip. '
                     'On iPhone: open it in the Google Drive app → ⋯ → <b>Send a copy</b> → <b>Save Video</b>.</p>'))


print("Starting. Clips appear below. Tap ■ to stop.\n")
env = dict(os.environ, HF_HUB_VERBOSITY="error", HF_HUB_DISABLE_PROGRESS_BARS="1", PYTHONWARNINGS="ignore")
# own session: Colab's stop button only reaches this cell, which then asks liveclip to finish its open clips
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env, start_new_session=True)
stopping = False
while True:
    try:
        line = p.stdout.readline()
        if not line:
            if p.poll() is not None: break
            continue
        line = line.rstrip()
        if "unauthenticated requests" in line or "[ctranslate2]" in line: continue
        print(line)
        m = re.search(r"saved (\S+\.mp4)$", line)
        if m and os.path.exists(m.group(1)): show(m.group(1))
    except KeyboardInterrupt:
        if not stopping:
            stopping = True; print("\nStopping… finishing the clips that are being made.")
            p.send_signal(signal.SIGINT)
        else:
            p.kill(); break
print("\nDone. If you lost this page, run the next cell to see every clip again.")

In [1]:
#@title 🎞 Show all my clips again
import glob, os, base64
from IPython.display import display, HTML
clips = sorted(glob.glob("/content/clips/*.mp4"))
if not clips: print("No clips yet. Run the cell above first.")
for path in clips:
    b64 = base64.b64encode(open(path, "rb").read()).decode(); name = os.path.basename(path)
    display(HTML(f'<p style="font:600 15px sans-serif;margin:14px 0 6px">🎬 {name}</p>'
                 f'<video controls playsinline style="width:100%;max-width:360px;border-radius:12px" src="data:video/mp4;base64,{b64}"></video>'
                 f'<br><a download="{name}" href="data:video/mp4;base64,{b64}" style="display:inline-block;margin-top:6px;'
                 f'padding:10px 16px;border-radius:10px;background:#e8684a;color:#fff;font:600 15px sans-serif;text-decoration:none">⬇ Download {name}</a>'))

No clips yet. Run the cell above first.


### Tips
- **Wrong part shown as the face?** Pick the corner in **facecam** and tap play again. Streams with no facecam: pick *no facecam*.
- **Live stream, no clips after 10 minutes?** The chat hasn't exploded yet. Keep it running, or paste a finished video instead.
- **Want it faster?** Menu → **Runtime → Change runtime type → T4 GPU** before you tap play. Captions then take a couple of seconds instead of ~30.
- **Phone screen locked or tab closed?** The clips stay on the Colab computer for a while. Come back and run *Show all my clips again*. With the Google Drive box ticked they're also safe in your Drive.
- **Download button does nothing on iPhone?** That's an iPhone limit. Tick **save_to_google_drive** and run again.
- Watch each clip before you post it. Captions can have mistakes; fix them in Edits or CapCut.
- Free Colab sessions stop after a few hours or when idle. That's Google's limit, not this tool's.